In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from preprocess import to_relative, normalize
from stroke_model import StrokeModel, StrokeDataset
from trainer import HandwritingTrainer
from handwriting_inference import Handwrite
from utils import plot_strokes

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

## 1. Data Preprocessing
- Loading raw stroke sample from collected data in Tkinter
- Apply Ramer-Douglas-Peucker algorithm to simplify it
- Convert absolute coordinates to relative
- Normalize the scale

In [ ]:
DATA_PATH = "./data/strokes.npy"
samples = np.load(DATA_PATH, allow_pickle = True)

raw_sample = samples[0]

print(f"Raw stroke sequence length: {len(raw_sample)}")

relative_sample = to_relative(raw_sample)
normalized_sample = normalize(relative_sample)

print(f"Processed sequence length (relative & normalized): {len(normalized_sample)}")

plot_strokes(normalized_sample, multiple = False)

## 2. Model Initialization
- Initializing the model
- Varaiational Autoencoder with an MDN output layer
- Quick overfitting test on a single batch to ensure the loss decreases and the KL Divergence is working

In [ ]:
single_sample = [normalized_sample]
dataset = StrokeDataset([single_sample[0] for _ in range(10)])
collate_fn = lambda batch: pad_sequence(batch, batch_first = True)

train_loader = DataLoader(
    dataset, 
    batch_size = 2, 
    shuffle = True, 
    collate_fn = collate_fn
)

model = StrokeModel(
    input_size=5, 
    hidden_size=256, 
    latent_size=64, 
    num_layers=1
)

trainer = HandwritingTrainer(
    model = model, 
    learning_rate = 0.001, 
    device = device
)

trainer.fit(
    train_loader = train_loader, 
    val_loader = train_loader,
    epochs = 5, 
    patience = 3, 
    checkpoint_path = "./models/demo_model.pth"
)

plt.plot(trainer.history["train_loss"], label="Train Loss")
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 3. Model Inference and Generation
- Loading in a pre-trained model
- Reconstruction and generating a completely new one from a latent vector

In [ ]:
model.load_state_dict(torch.load("./models/handwriting_model.pth", map_location = device))
generator = Handwrite(model = model, device = device)

sample_tensor = dataset[0]

print("Reconstructing ground truth...")

reconstructed_strokes = generator.reconstruct(sample_tensor)
plot_strokes(reconstructed_strokes, multiple = False)

print("Generating new handwriting freely...")

generated_strokes = generator.generate(sample_tensor, max_steps = 150)
plot_strokes(generated_strokes, multiple = False)